### 1. Preparação do Ambiente e Importação de Bibliotecas
Nesta célula inicial, configuramos o ambiente de trabalho carregando as bibliotecas essenciais para a Ciência de Dados e Machine Learning:

* **Pandas e Numpy**: Fundamentais para a manipulação de matrizes e tabelas de dados biomecânicos.
* **Scikit-Learn**: Fornece as ferramentas de divisão de dados (`train_test_split`), normalização (`StandardScaler`) e as métricas de validação.
* **XGBoost e Random Forest**: Os algoritmos de inteligência artificial que serão treinados para identificar o risco de entorse.
* **OS e Time**: Bibliotecas de sistema para gestão de ficheiros e medição da eficiência (tempo de execução) dos modelos.

In [2]:
# ==============================================================================
# CÉLULA 1: IMPORTAÇÃO DE BIBLIOTECAS E CONFIGURAÇÃO DA SEMENTE GLOBAL
# Responsabilidade: Carregar ferramentas de Manipulação, ML, Deep Learning e Gráficos.
# ==============================================================================
import pandas as pd
import numpy as np
import os
import time
import random

# 1. Visualização de Dados (Para os Gráficos de Importância e Matrizes)
import matplotlib.pyplot as plt
import seaborn as sns

# 2. Machine Learning Tradicional e Validação Cruzada (Scikit-Learn)
# CORREÇÃO: Removidos espaços invisíveis que causavam SyntaxError e adicionado StratifiedKFold
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, recall_score, f1_score, 
                             confusion_matrix, classification_report, roc_auc_score)
from sklearn.ensemble import RandomForestClassifier

# 3. Modelos de Elite (XGBoost)
try:
    from xgboost import XGBClassifier
    print("✅ XGBoost: Carregado.")
except ImportError:
    print("⚠️ XGBoost não instalado. Use: !pip install xgboost")

# 4. Deep Learning (PyTorch)
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    print("✅ PyTorch: Carregado.")
except ImportError:
    print("⚠️ PyTorch não instalado. Use: !pip install torch")

# ==============================================================================
# CONFIGURAÇÕES DE REPRODUTIBILIDADE CIENTÍFICA (CRUCIAL PARA O JÚRI)
# ==============================================================================
def definir_semente_global(seed=42):
    """Garante que os resultados da Rede Neural e Árvores sejam idênticos em cada execução."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # Configurações específicas para o PyTorch
    if 'torch' in globals():
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # Se usar múltiplas GPUs
        # Força o PyTorch a usar algoritmos determinísticos (essencial para artigos)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

definir_semente_global(42)

# Configurações Estéticas para o Relatório/Artigo
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100  # Melhora a resolução dos gráficos para colar no Word

print("🚀 Todas as bibliotecas foram carregadas e as sementes foram fixadas com sucesso!")

✅ XGBoost: Carregado.
✅ PyTorch: Carregado.
🚀 Todas as bibliotecas foram carregadas e as sementes foram fixadas com sucesso!


### 2. Carregamento e Consolidação dos Dados Brutos
Nesta etapa, o sistema acede à diretoria externa `Data` para importar os conjuntos de dados biomecânicos provenientes dos três artigos científicos base. 

* **Caminhos Relativos**: Utilizamos a biblioteca `os` para garantir que o código seja portátil entre diferentes sistemas operativos (Windows/Mac/Linux).
* **Tratamento de Exceções**: Implementámos um bloco `try-except` para capturar erros de ficheiro não encontrado, garantindo que o programa não falhe abruptamente caso a pasta `Data` esteja ausente ou mal posicionada.
* **Validação Inicial**: O sistema confirma o sucesso do carregamento exibindo o volume total de amostras prontas para processamento.

In [3]:
# ==============================================================================
# CÉLULA 2: CARREGAMENTO DE DADOS (CAMINHO CORRIGIDO)
# Responsabilidade: Carregar o dataset unificado do caminho exato do Desktop.
# ==============================================================================
import os
import pandas as pd

print("--- FASE 1: CARREGAMENTO ---")

# Caminho exato e direto baseado no seu print do Windows
BASE_PATH = r"C:\Users\User\Desktop\escola\Projeto\PlaySafe4All\PlaySafe4All\Data"
NOME_FICHEIRO = "Dataset_Unificado_Final.csv"
CAMINHO_COMPLETO = os.path.join(BASE_PATH, NOME_FICHEIRO)

load_successful = False

try:
    # Mudámos o encoding para 'latin1' para aceitar os acentos e cedilhas do Excel do Windows
    df_unificado = pd.read_csv(CAMINHO_COMPLETO, sep=None, engine='python', encoding='latin1')
    
    print(f"✅ Ficheiro '{NOME_FICHEIRO}' carregado com sucesso!")
    print(f"   ↳ Linhas (Atletas): {df_unificado.shape[0]}")
    print(f"   ↳ Colunas (Atributos): {df_unificado.shape[1]}")
    
    load_successful = True

except FileNotFoundError:
    print(f"❌ ERRO: Ainda não encontrei o ficheiro.")
    print(f"   Caminho tentado: {CAMINHO_COMPLETO}")
    print("   💡 Verifique se o utilizador do Windows é mesmo 'User' ou se tem outro nome.")
    load_successful = False
    
except Exception as e:
    print(f"❌ ERRO ao ler o ficheiro: {str(e)}")
    load_successful = False

--- FASE 1: CARREGAMENTO ---
✅ Ficheiro 'Dataset_Unificado_Final.csv' carregado com sucesso!
   ↳ Linhas (Atletas): 61
   ↳ Colunas (Atributos): 13


# 3. Pré-Processamento Dinâmico e Configuração da Validação Cruzada

Esta célula executa o isolamento da variável-alvo (`TARGET_INJURY`) e realiza o pré-processamento adaptativo da matriz de atributos preditores ($X$). O objetivo principal deste bloco é criar uma estrutura de dados flexível e automatizada que prepare todas as variáveis disponíveis no *dataset* para os modelos, mantendo a blindagem contra falhas estatísticas.

###  Operações Realizadas:
* **Remoção de Identificadores:** Exclusão de colunas nominativas ou de indexação (`Numero`, `ID`, `Atleta`) para evitar o enviesamento do algoritmo com dados não-físicos.
* **Imputação por Mediana:** Tratamento automático de eventuais dados em falta (`NaN`) utilizando a mediana de cada coluna preditora, preservando a robustez estatística do conjunto.
* **Codificação Categórica (One-Hot Encoding):** Aplicação da função `pd.get_dummies(drop_first=True)` para detetar e converter automaticamente variáveis de texto ou categóricas em representações binárias numéricas, expandindo a compatibilidade da matriz para algoritmos de regressão e árvores.

###  Prevenção de *Data Leakage* e Amostragem Estratificada:
O pipeline foi desenhado para **não aplicar qualquer escalonamento global nesta fase**. A transformação de escala através do `StandardScaler` é intencionalmente adiada para ser executada de forma independente dentro do ciclo de cada *fold*. Isto garante que os parâmetros estatísticos do grupo de teste nunca interfiram no processo de treino.

A estabilidade da avaliação é assegurada com a configuração da **Validação Cruzada Estratificada em 10 Blocos (Stratified 10-Fold CV)**. Esta estratégia garante que a proporção exata entre atletas saudáveis (classe maioritária) e em risco de lesão (classe minoritária) seja respeitada em todas as divisões de treino e teste, mitigando o forte desequilíbrio nativo dos dados desportivos e fornecendo métricas de generalização altamente realistas.

In [4]:
# ==============================================================================
# CÉLULA 3: PRÉ-PROCESSAMENTO (SINCRONIZADO COM O SEU DATASET)
# Responsabilidade: Preparar a matriz X e o vetor y usando TARGET_INJURY diretamente.
# ==============================================================================
from sklearn.model_selection import StratifiedKFold
import pandas as pd

print("\n--- FASE 2: PREPARAÇÃO DE DADOS (ABORDAGEM CROSS-VALIDATION) ---")

# 1. Usar o DataFrame unificado que carregámos na Célula 2
df = df_unificado.copy()

# Remover colunas de identificação se existirem
for col_to_drop in ['Numero', 'ID', 'Atleta']:
    if col_to_drop in df.columns:
        df.drop(columns=[col_to_drop], inplace=True)

# 2. Definição do Alvo com o nome exato do seu print
TARGET_COL = 'TARGET_INJURY'

if TARGET_COL not in df.columns:
    raise KeyError(f"❌ ERRO: A coluna '{TARGET_COL}' não foi encontrada. Verifique se o nome tem espaços ou letras minúsculas.")

# Definir as colunas preditoras (as que restam no DataFrame excluindo o Alvo)
cols_preditoras = [col for col in df.columns if col != TARGET_COL]
print(f"🔄 Atributos preditores detetados: {len(cols_preditoras)}")

# Preencher quaisquer valores em falta (NaN) com a mediana das colunas preditoras
df[cols_preditoras] = df[cols_preditoras].fillna(df[cols_preditoras].median(numeric_only=True))

# 3. Definição Direta de X e y
y = df[TARGET_COL].astype(int) # Garante que está no formato numérico inteiro
X = df.drop(columns=[TARGET_COL])

# Aplicar One-Hot Encoding caso existam variáveis de texto/categóricas nas preditoras
X = pd.get_dummies(X, drop_first=True)

# 4. Configuração da Validação Cruzada Estratificada (10 Folds)
cv_estrategia = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

print(f"✅ Dados Prontos para Validação Cruzada!")
print(f"   ↳ Matriz Completa de Atributos (X): {X.shape}")
print(f"   ↳ Vetor de Alvos (y):               {y.shape}")
print(f"   ↳ Distribuição de Classes (0/1):    {dict(y.value_counts())}")


--- FASE 2: PREPARAÇÃO DE DADOS (ABORDAGEM CROSS-VALIDATION) ---
🔄 Atributos preditores detetados: 11
✅ Dados Prontos para Validação Cruzada!
   ↳ Matriz Completa de Atributos (X): (61, 270)
   ↳ Vetor de Alvos (y):               (61,)
   ↳ Distribuição de Classes (0/1):    {0: np.int64(46), 1: np.int64(15)}


# 4. Modelação com XGBoost e Otimização por Gradient Boosting

Esta célula executa a fase de treino e avaliação do algoritmo **XGBoost (eXtreme Gradient Boosting)** sob a infraestrutura da Validação Cruzada Estratificada em 10 Blocos. O objetivo é construir um modelo baseado em árvores de decisão sequenciais que aprenda com os erros dos estimadores anteriores, maximizando a taxa de acerto clínico sem comprometer a estabilidade estatística.

###  Estratégias de Regularização e Hiperparâmetros Controlados:
Para prevenir o risco latente de sobreajuste (*overfitting*), ao qual os modelos de *boosting* são propensos em amostras de dimensão controlada, foram aplicadas restrições geométricas severas:
* **Taxa de Aprendizagem Moderada (`learning_rate=0.05`):** Funciona como um fator de encolhimento (*shrinkage*) que reduz a magnitude de atualização dos pesos a cada nova árvore, forçando uma convergência mais lenta, ponderada e robusta.
* **Profundidade Construtiva (`max_depth=3`):** Limita o crescimento vertical de cada árvore de decisão a apenas três níveis. Isto cria estimadores propositadamente fracos (*weak learners*), impedindo que o algoritmo desenhe ramificações excessivamente complexas ou decore o ruído nativo do *dataset*.
* **Controlo de Iterações (`n_estimators=100`):** Fixa o teto de expansão do modelo a 100 árvores sequenciais, um limite seguro que bloqueia o esforço computacional antes que ocorra a memorização da amostra.

###  Escalonamento e Proteção dos Folds:
A normalização através do `StandardScaler` é instanciada e executada de forma **estritamente isolada dentro de cada fold**. O cálculo dos parâmetros de média e desvio padrão ocorre exclusivamente sobre a matriz `X_fold_train_scaled`, garantindo uma barreira de segurança (*firewall* estatística) que impede a fuga de informação (*Data Leakage*) para os dados de teste.

Ao fim das 10 iterações, os resultados parciais de *Accuracy*, *F1-Score* e *AUC-ROC* são consolidados matematicamente por meio das suas médias e desvios padrão reais, acumulando a matriz de confusão para uma auditoria direta aos acertos e erros clínicos do sistema.

In [5]:
# ==============================================================================
# CÉLULA 4: MODELO XGBOOST (VALIDAÇÃO CRUZADA PROFISSIONAL)
# Responsabilidade: Treinar e avaliar o XGBoost usando a estratégia de 10 Folds.
# ==============================================================================
import numpy as np
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

print("\n--- FASE 3: XGBOOST (GRADIENT BOOSTING COM CROSS-VALIDATION) ---")

# Verificação de segurança: Confirmar se a estratégia da Célula 3 está ativa
if 'cv_estrategia' not in locals() or 'X' not in locals():
    print("❌ ERRO: Precisa de correr a CÉLULA 3 reformulada primeiro!")
else:
    # 1. Configuração do Modelo XGBoost (Hiperparâmetros controlados para evitar Overfitting)
    modelo_xgb = XGBClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
        eval_metric='logloss'
    )

    # Listas para acumular os resultados de cada um dos 10 Folds
    historico_accuracy = []
    historico_f1 = []
    historico_auc = []
    
    # Matriz acumulada para somar todas as matrizes de confusão no final
    matriz_confusao_total = np.zeros((2, 2), dtype=int)

    print("🚀 A iniciar os 10 ciclos de Treino e Teste (Stratified 10-Fold)...")

    # 2. O Loop da Validação Cruzada (Garante escalonamento seguro e sem Data Leakage)
    for fold, (idx_treino, idx_teste) in enumerate(cv_estrategia.split(X, y), 1):
        
        # Divisão dos dados do Fold atual (Trabalha com .iloc porque X é um DataFrame)
        X_fold_train, X_fold_test = X.iloc[idx_treino], X.iloc[idx_teste]
        y_fold_train, y_fold_test = y.iloc[idx_treino], y.iloc[idx_teste]
        
        # ESCALONAMENTO SEGURO: Ajusta o scaler APENAS nos dados de treino deste fold
        scaler = StandardScaler()
        X_fold_train_scaled = scaler.fit_transform(X_fold_train)
        X_fold_test_scaled = scaler.transform(X_fold_test)
        
        # Treinar o modelo neste bloco de dados
        modelo_xgb.fit(X_fold_train_scaled, y_fold_train)
        
        # Fazer as previsões de classes (0 ou 1) e probabilidades (para o AUC)
        y_pred = modelo_xgb.predict(X_fold_test_scaled)
        y_proba = modelo_xgb.predict_proba(X_fold_test_scaled)[:, 1]
        
        # Calcular e guardar as métricas deste fold específico
        historico_accuracy.append(accuracy_score(y_fold_test, y_pred))
        historico_f1.append(f1_score(y_fold_test, y_pred, zero_division=0))
        historico_auc.append(roc_auc_score(y_fold_test, y_proba))
        
        # Somar a matriz de confusão deste fold à matriz global
        matriz_confusao_total += confusion_matrix(y_fold_test, y_pred)

    # 3. Apresentação dos Resultados Médios Finais
    print("\n" + "="*50)
    print("📊 RESULTADOS CONSOLIDADOS (MÉDIA DOS 10 FOLDS)")
    print("="*50)
    print(f"✅ Precisão Média (Accuracy): {np.mean(historico_accuracy)*100:.2f}% (± {np.std(historico_accuracy)*100:.1f}%)")
    print(f"✅ F1-Score Médio:            {np.mean(historico_f1)*100:.2f}%")
    print(f"✅ AUC-ROC Médio:             {np.mean(historico_auc)*100:.2f}%")
    
    print("\n🧩 MATRIZ DE CONFUSÃO CONSOLIDADA (SOMA DE TODOS OS ATLETAS):")
    print(matriz_confusao_total)
    print(f"   ↳ Verdadeiros Negativos (Saudáveis previstos corretamente): {matriz_confusao_total[0,0]}")
    print(f"   ↳ Falsos Positivos (Previu lesão mas o atleta estava bem):  {matriz_confusao_total[0,1]}")
    print(f"   ↳ Falsos Negativos (⚠️ FALHOU A LESÃO DO ATLETA):           {matriz_confusao_total[1,0]}")
    print(f"   ↳ Verdadeiros Positivos (Lesões previstas com sucesso!):    {matriz_confusao_total[1,1]}")


--- FASE 3: XGBOOST (GRADIENT BOOSTING COM CROSS-VALIDATION) ---
🚀 A iniciar os 10 ciclos de Treino e Teste (Stratified 10-Fold)...

📊 RESULTADOS CONSOLIDADOS (MÉDIA DOS 10 FOLDS)
✅ Precisão Média (Accuracy): 72.38% (± 14.1%)
✅ F1-Score Médio:            21.67%
✅ AUC-ROC Médio:             67.75%

🧩 MATRIZ DE CONFUSÃO CONSOLIDADA (SOMA DE TODOS OS ATLETAS):
[[41  5]
 [12  3]]
   ↳ Verdadeiros Negativos (Saudáveis previstos corretamente): 41
   ↳ Falsos Positivos (Previu lesão mas o atleta estava bem):  5
   ↳ Falsos Negativos (⚠️ FALHOU A LESÃO DO ATLETA):           12
   ↳ Verdadeiros Positivos (Lesões previstas com sucesso!):    3
